Create a customer/orders analytical pipeline with joins and aggregations; validate record counts.

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark=SparkSession.builder.appName("join_agg_task").master("local[*]").getOrCreate()

In [ ]:
customers=spark.createDataFrame([
    (101 , "Ram" , 25),
    (102  , "Sita" , 24),
    (103 , "Ravan" , 23),
    (104 , "Laksham" , 22)
], ["customer_id" , "name" , "age"])

orders=spark.createDataFrame(
    [
        (1 , 101 , "laptop" , 34000) ,
        (2 , 103 , "phone" , 35000),
        (3 , 103 , "earrings" , 300),
        (4 , 104 , "table" , 3000)
    ], ["order_id" , "customer_id" , "product_name" , "price"]
)

In [ ]:
customers.show()

+-----------+-------+---+
|customer_id|   name|age|
+-----------+-------+---+
|        101|    Ram| 25|
|        102|   Sita| 24|
|        103|  Ravan| 23|
|        104|Laksham| 22|
+-----------+-------+---+



In [ ]:
orders.show()

+--------+-----------+------------+-----+
|order_id|customer_id|product_name|price|
+--------+-----------+------------+-----+
|       1|        101|      laptop|34000|
|       2|        103|       phone|35000|
|       3|        103|    earrings|  300|
|       4|        104|       table| 3000|
+--------+-----------+------------+-----+



All the customers data who have placed an order with their order details

In [ ]:
customers.join(orders , "customer_id").show()

+-----------+-------+---+--------+------------+-----+
|customer_id|   name|age|order_id|product_name|price|
+-----------+-------+---+--------+------------+-----+
|        101|    Ram| 25|       1|      laptop|34000|
|        103|  Ravan| 23|       2|       phone|35000|
|        103|  Ravan| 23|       3|    earrings|  300|
|        104|Laksham| 22|       4|       table| 3000|
+-----------+-------+---+--------+------------+-----+



All the customers data even if he placed an order or not

In [ ]:
customers.join(orders , "customer_id" , "left").show()

+-----------+-------+---+--------+------------+-----+
|customer_id|   name|age|order_id|product_name|price|
+-----------+-------+---+--------+------------+-----+
|        101|    Ram| 25|       1|      laptop|34000|
|        102|   Sita| 24|    NULL|        NULL| NULL|
|        103|  Ravan| 23|       3|    earrings|  300|
|        103|  Ravan| 23|       2|       phone|35000|
|        104|Laksham| 22|       4|       table| 3000|
+-----------+-------+---+--------+------------+-----+



SUM OF ORDER prices, they have placed.

In [ ]:
customers.join(orders , "customer_id").groupBy("name").sum("price").show()

+-------+----------+
|   name|sum(price)|
+-------+----------+
|  Ravan|     35300|
|Laksham|      3000|
|    Ram|     34000|
+-------+----------+



Want to see the sum of prices of orders for the customer\'s who have\'nt even ordered a thing with null as value

In [ ]:
import pyspark.sql.functions as sf

In [ ]:
customers.join(orders , "customer_id" , "left").groupBy("customer_id").agg(sf.sum("price").alias("total_price")).show()

+-----------+-----------+
|customer_id|total_price|
+-----------+-----------+
|        101|      34000|
|        102|       NULL|
|        103|      35300|
|        104|       3000|
+-----------+-----------+



I want to see the the sum of no order people to have 0 as value in the total price instead of null

In [ ]:
customers.join(orders , "customer_id" , "left").groupBy("customer_id").agg(sf.coalesce(sf.sum("price"), sf.lit(0)).alias("total_price")).show()

+-----------+-----------+
|customer_id|total_price|
+-----------+-----------+
|        101|      34000|
|        102|          0|
|        103|      35300|
|        104|       3000|
+-----------+-----------+

